# MovieMind: Exploratory Data Analysis (EDA)

In this notebook, we perform an initial exploratory data analysis on the raw TMDB 5000 movies and credits datasets. Our goal is to understand the structure, identify missing values, check for duplicates, and explore the distributions of key features without modifying or cleaning the data yet.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use('ggplot')

## 1. Data Loading
We load the two raw datasets: `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv`.

In [ ]:
movies = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

## 2. Dataset Overview
Let's check the shape, columns, first and last rows, data types, and basic statistics for the movies dataset.

In [ ]:
print('Movies Shape:', movies.shape)
print('Movies Columns:', list(movies.columns))

In [ ]:
movies.head()

In [ ]:
movies.tail()

In [ ]:
movies.info()

In [ ]:
movies.describe()

Now, let's examine the credits dataset.

In [ ]:
print('Credits Shape:', credits.shape)
print('Credits Columns:', list(credits.columns))

In [ ]:
credits.head()

In [ ]:
credits.tail()

In [ ]:
credits.info()

## 3. Missing Values and Duplicates
We check for null values and duplicate rows across both datasets to understand data cleanliness.

In [ ]:
missing_movies = movies.isnull().sum()
missing_pct_movies = (missing_movies / len(movies)) * 100
pd.DataFrame({'Missing Count': missing_movies, 'Percentage': missing_pct_movies})

In [ ]:
missing_credits = credits.isnull().sum()
missing_pct_credits = (missing_credits / len(credits)) * 100
pd.DataFrame({'Missing Count': missing_credits, 'Percentage': missing_pct_credits})

In [ ]:
print('Movies duplicates:', movies.duplicated().sum())
print('Credits duplicates:', credits.duplicated().sum())

## 4. Dataset Relationship & Uniqueness
We determine how the two datasets can be merged. The movies dataset has an `id` column, and the credits dataset has a `movie_id` column. We will verify if they contain unique identifiers for each row.

In [ ]:
print('Movies unique IDs:', movies['id'].nunique())
print('Credits unique IDs:', credits['movie_id'].nunique())
print('Are movie IDs unique?', movies['id'].nunique() == len(movies))

Since both datasets contain exactly 4,803 unique rows and there is a perfect 1:1 relationship between `id` and `movie_id`, this column can be used to join them.

## 5. Visualizations
Let's visualize some of the numeric columns like vote average, popularity, and release year to understand their distributions.

In [ ]:
# Extract year from release_date
movies['release_year'] = pd.to_datetime(movies['release_date'], errors='coerce').dt.year

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of Ratings
movies['vote_average'].plot(kind='hist', bins=30, ax=axes[0,0], title='Distribution of Movie Ratings (vote_average)', color='skyblue', edgecolor='black')

# Distribution of Vote Counts
movies['vote_count'].plot(kind='hist', bins=30, ax=axes[0,1], title='Distribution of Vote Counts', color='lightgreen', edgecolor='black', logy=True)

# Movies by Release Year
movies['release_year'].plot(kind='hist', bins=50, ax=axes[1,0], title='Movies by Release Year', color='coral', edgecolor='black')

# Popularity Distribution
movies['popularity'].plot(kind='hist', bins=50, ax=axes[1,1], title='Popularity Distribution', color='gold', edgecolor='black', logy=True)

plt.tight_layout()
plt.show()

## EDA Findings and Next Steps

### Findings
- **Dataset Sizes**: Both datasets contain exactly 4,803 rows. The `movies` dataset has 20 columns, and `credits` has 4 columns.
- **Important Columns**:
  - `movies`: `id`, `title`, `genres`, `keywords`, `overview`, `popularity`, `vote_average`, `vote_count`, `release_date`.
  - `credits`: `movie_id`, `title`, `cast`, `crew`.
- **Missing Values**:
  - The `credits` dataset has no missing values.
  - The `movies` dataset has missing values in: `homepage` (3091), `tagline` (844), `overview` (4), `runtime` (2), and `release_date` (1).
- **Duplicates**: There are no duplicate rows in either dataset.
- **Dataset Relationship**: Both datasets use a unique identifier for movies (`id` in the movies dataset and `movie_id` in the credits dataset). Both columns have 4,803 unique values, meaning there is a perfect 1:1 relationship. This is the column we will use to join the datasets.
- **Recommendation Features**: Features like `genres`, `keywords`, `overview`, `cast`, and `crew` are currently stored as JSON string arrays. These will be highly useful for content-based recommendations but will require parsing.

### Recommended Next Steps (Preprocessing)
1. **Merge Datasets**: Join `movies` and `credits` on the `id` / `movie_id` column.
2. **Handle Missing Values**: Fill missing text fields (like `overview`) with empty strings or drop unusable rows.
3. **Parse JSON Columns**: Convert the JSON strings in `genres`, `keywords`, `cast`, and `crew` into workable lists of strings (e.g., extracting just the genre names or the top 3 actors).
4. **Feature Selection**: Keep only the columns relevant for building the recommendation engine (e.g., `movie_id`, `title`, `overview`, `genres`, `keywords`, `cast`, `crew`).
5. **Save Processed Data**: Save the cleaned and merged dataset to `data/processed/` for the modeling phase.

---
# 6. Data Preprocessing & Feature Engineering

Now that we have explored the dataset, we will proceed to clean and format it for our recommendation engine. We will merge the datasets, extract text from JSON strings, normalize strings, and create a single `tags` column which will serve as the core feature for calculating content-based similarities.

## 6.1. Merging and Column Selection
We will merge the datasets on `id` and `movie_id`, and then select only the columns relevant for building recommendations and displaying movie cards.

In [ ]:
movies = pd.read_csv('../data/raw/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/raw/tmdb_5000_credits.csv')

# Merge datasets
df = movies.merge(credits, left_on='id', right_on='movie_id')

# Verify row count
print(f"Merged row count: {len(df)} (Expected: 4803)")

# Select useful columns
useful_cols = [
    'id', 'title_x', 'overview', 'genres', 'keywords', 'cast', 'crew', 
    'popularity', 'vote_average', 'vote_count', 'release_date'
]
df = df[useful_cols]
df = df.rename(columns={'title_x': 'title'})

# Drop rows with missing id or title
df = df.dropna(subset=['id', 'title'])
df.head(2)

## 6.2. JSON Parsing and Feature Extraction
The `genres`, `keywords`, `cast`, and `crew` columns are currently string representations of JSON arrays. We will use `ast.literal_eval` to parse them into Python lists of dictionaries. Then we extract:
- All genre names
- All keyword names
- The top 3 actors from the cast
- The director from the crew

In [ ]:
import ast

def extract_names(obj_str):
    try:
        L = ast.literal_eval(obj_str)
        return [i['name'] for i in L]
    except:
        return []

def extract_top_3_cast(obj_str):
    try:
        L = ast.literal_eval(obj_str)
        return [i['name'] for i in L[:3]]
    except:
        return []

def extract_director(obj_str):
    try:
        L = ast.literal_eval(obj_str)
        for i in L:
            if i['job'] == 'Director':
                return [i['name']]
        return []
    except:
        return []

# Apply extractors
df['genres'] = df['genres'].apply(extract_names)
df['keywords'] = df['keywords'].apply(extract_names)
df['cast'] = df['cast'].apply(extract_top_3_cast)
df['crew'] = df['crew'].apply(extract_director)

df.head(2)

## 6.3. Data Cleaning & Text Normalization
Next, we handle missing data and normalize the text fields (lowercase and remove spaces inside compound names so that "Science Fiction" becomes "sciencefiction" or similar, though for now we just clean the strings for the overall tags).

In [ ]:
import re

# Convert missing overview to empty string
df['overview'] = df['overview'].fillna("")

# Extract release year
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year

# Normalizer function
def normalize_list(lst):
    return [re.sub(r"[^a-zA-Z0-9]", "", str(x).lower()) for x in lst]

# We remove spaces between names in cast/crew/genres to treat them as single entities
# e.g. "Johnny Depp" -> "johnnydepp"
df['genres_norm'] = df['genres'].apply(normalize_list)
df['keywords_norm'] = df['keywords'].apply(normalize_list)
df['cast_norm'] = df['cast'].apply(normalize_list)
df['crew_norm'] = df['crew'].apply(normalize_list)

# Clean overview (lowercase, remove punctuation)
df['overview_clean'] = df['overview'].apply(lambda x: re.sub(r"[^a-zA-Z0-9\s]", "", str(x).lower()))


## 6.4. Creating the `tags` Column
We combine `overview`, `genres`, `keywords`, `cast`, and `crew` (director) into a single string called `tags`. This will be used to compute document similarities later.

In [ ]:
# Combine lists into strings with spaces
df['tags'] = df['overview_clean'] + " " + \
             df['genres_norm'].apply(lambda x: " ".join(x)) + " " + \
             df['keywords_norm'].apply(lambda x: " ".join(x)) + " " + \
             df['cast_norm'].apply(lambda x: " ".join(x)) + " " + \
             df['crew_norm'].apply(lambda x: " ".join(x))

# Normalize any extra whitespace
df['tags'] = df['tags'].apply(lambda x: " ".join(x.split()))

print("Sample tags:\n", df['tags'].iloc[0])

## 6.5. Final Checks and Export
We check the final shape, duplicates, and missing values before exporting the processed dataset.

In [ ]:
# Final columns to keep
final_cols = ['id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 
              'popularity', 'vote_average', 'vote_count', 'release_date', 'release_year', 'tags']
processed_df = df[final_cols]

print("Final Shape:", processed_df.shape)
print("Missing Values:\n", processed_df.isnull().sum())
print("Duplicate IDs:", processed_df['id'].duplicated().sum())

# Save to processed data folder
processed_df.to_csv('../data/processed/movies_processed.csv', index=False)
print("Saved processed dataset to ../data/processed/movies_processed.csv")

## Preprocessing Summary
- **Original row count**: 4,803 (movies), 4,803 (credits)
- **Merged row count**: 4,803
- **Final row count**: 4,803
- **Columns retained**: `id`, `title`, `overview`, `genres`, `keywords`, `cast`, `crew`, `popularity`, `vote_average`, `vote_count`, `release_date`, `release_year`, `tags`
- **Columns extracted from JSON**: `genres` (all), `keywords` (all), `cast` (top 3), `crew` (director)
- **Missing-value handling**: `overview` filled with empty strings. `release_year` left as NaN where `release_date` was missing. Text inputs normalized.
- **Purpose of the `tags` column**: To serve as a single concatenated text feature combining the plot, genres, tags, top cast, and director for downstream TF-IDF and cosine similarity processing.
- **Path of the processed dataset**: `data/processed/movies_processed.csv`